# Multi-Model Benchmarking & Holdout Backtesting
### Applied Time Series Analysis (ATSA) Educational Notebook

This tutorial demonstrates:
1. **Out-of-sample train/test holdout backtesting**.
2. **Benchmarking against a Naive persistence random-walk baseline**.
3. **Comparing ARIMA vs SARIMA vs Holt-Winters vs Moving Average**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error

# 1. Load Monthly Sales Synthetic Data
np.random.seed(42)
n = 72
t = np.arange(n)
y = 100 + 1.2 * t + 8 * np.sin(2 * np.pi * t / 12) + np.random.normal(0, 2, n)
df = pd.Series(y, index=pd.date_range('2018-01-01', periods=n, freq='MS'))

# Train/Test Split (80% Train, 20% Test)
train_size = int(n * 0.8)
train, test = df.iloc[:train_size], df.iloc[train_size:]
print(f"Train observations: {len(train)}, Test observations: {len(test)}")

## 2. Fit Candidate Models & Evaluate on Test Set

In [ ]:
models = {}
# Model 1: Naive Random Walk Benchmark
naive_forecast = np.full(len(test), train.iloc[-1])
models['Naive Baseline'] = naive_forecast

# Model 2: Moving Average (window=6)
ma_forecast = np.full(len(test), train.tail(6).mean())
models['Moving Average (6)'] = ma_forecast

# Model 3: ARIMA(1,1,1)
arima_fit = ARIMA(train, order=(1,1,1)).fit()
models['ARIMA(1,1,1)'] = arima_fit.forecast(steps=len(test)).values

# Model 4: Holt-Winters Exponential Smoothing
hw_fit = ExponentialSmoothing(train, trend='add', seasonal='add', seasonal_periods=12).fit()
models['Holt-Winters'] = hw_fit.forecast(steps=len(test)).values

# Leaderboard evaluation
leaderboard = []
for name, pred in models.items():
    rmse = np.sqrt(mean_squared_error(test.values, pred))
    mae = mean_absolute_error(test.values, pred)
    mape = np.mean(np.abs((test.values - pred) / test.values)) * 100
    leaderboard.append({'Model': name, 'RMSE': round(rmse, 2), 'MAE': round(mae, 2), 'MAPE (%)': round(mape, 2)})

leaderboard_df = pd.DataFrame(leaderboard).sort_values(by='RMSE')
print(leaderboard_df.to_string(index=False))